In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# =====================================================================
# 1. 데이터셋 활용 및 삭제 (Drop)
# =====================================================================
df = pd.read_csv('Clean_Dataset.csv')

# [피드백 1번 반영] 과적합을 유발하는 편명(flight)과 불필요한 인덱스 삭제
columns_to_drop = []
if 'Unnamed: 0' in df.columns:
    columns_to_drop.append('Unnamed: 0')
if 'flight' in df.columns:
    columns_to_drop.append('flight')
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# [피드백 3번 반영] 오류가 있던 season(성수기/비수기) 로직은 생성하지 않고 완전히 삭제함


# =====================================================================
# 3. 변수 생성 (Feature Engineering)
# =====================================================================
# ① 노선(Route) 변수 생성: 출발지_도착지
df['route'] = df['source_city'] + "_" + df['destination_city']
# route 생성 후 원래 도시 컬럼은 제거
df.drop(columns=['source_city', 'destination_city'], inplace=True)

# ② 예약 시점 구간 변수 생성
# [피드백 2번 반영] -1부터 시작하여 0일 데이터 손실(결측치) 완벽 방지
bins = [-1, 7, 21, df['days_left'].max()]
labels = ['1~7일 전(출발 직전)', '8~21일 전(단기)', '22일 이상(장기)']
df['booking_period'] = pd.cut(df['days_left'], bins=bins, labels=labels)

# ③ 타겟 변수 로그 변환 (가격 변동성 학습 최적화)
df['price'] = np.log1p(df['price'])


# =====================================================================
# 2. 숫자로 변환 (Encoding)
# =====================================================================
# ① class (절대 지우지 않고 0과 1로 매핑)
df['class'] = df['class'].map({'Economy': 0, 'Business': 1})

# ② stops: 직항 0, 1회 경유 1, 2회 이상 2
df['stops'] = df['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})

# ③ 도시 이름 원-핫 인코딩 (6개 도시 분할)
df = pd.get_dummies(df, drop_first=True)

# =====================================================================
# 4. 데이터 분리 (Train/Test Split - 8:2 비율)
# =====================================================================
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("=== 최종 전처리 및 데이터 분리 완료 ===")
print(f"X_train 데이터 크기: {X_train.shape}")
print(f"y_train 데이터 크기: {y_train.shape}")
print(f"X_test 데이터 크기: {X_test.shape}")
print(f"y_test 데이터 크기: {y_test.shape}")
# =====================================================================
# 5. CSV 파일로 저장하기 (눈으로 확인하기 위한 용도)
# =====================================================================

# 1. 전체 전처리 완료된 데이터 하나로 저장하기
df.to_csv('final_preprocessed_data.csv', index=False, encoding='utf-8-sig')

# 2. (선택사항) 머신러닝용으로 분리된 Train / Test 데이터를 각각 저장하기
# 정답(y) 컬럼을 다시 붙여서 보기 좋게 하나의 파일로 만듭니다.
train_df = pd.concat([X_train, y_train], axis=1)
train_df.to_csv('train_data.csv', index=False, encoding='utf-8-sig')

test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False, encoding='utf-8-sig')

print("CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.")

=== 최종 전처리 및 데이터 분리 완료 ===
X_train 데이터 크기: (240122, 50)
y_train 데이터 크기: (240122,)
X_test 데이터 크기: (60031, 50)
y_test 데이터 크기: (60031,)
CSV 파일 저장이 완료되었습니다. 폴더를 확인해 주세요.


In [7]:
# Economy / Business 데이터 분리
df_economy = df[df['class'] == 0].copy()
df_business = df[df['class'] == 1].copy()

print("Economy shape:", df_economy.shape)
print("Business shape:", df_business.shape)

Economy shape: (206666, 51)
Business shape: (93487, 51)


In [13]:
#  ============================
#  1. Economy / Business 데이터 분리
#  ============================
df_economy = df[df['class'] == 0].copy()
df_business = df[df['class'] == 1].copy()

#  ============================
#  2. Economy: X, y 분리
#  ============================
X_eco = df_economy.drop(columns=['price', 'class'])
y_eco = df_economy['price']

# 상수 컬럼 제거
nunique_eco = X_eco.nunique()
constant_cols_eco = nunique_eco[nunique_eco <= 1].index
X_eco = X_eco.drop(columns=constant_cols_eco)

# 더미변수 중복 제거
prefixes = ['airline_', 'route_', 'departure_time_', 'arrival_time_']
for prefix in prefixes:
    cols = [c for c in X_eco.columns if c.startswith(prefix)]
    if len(cols) > 0:
         X_eco = X_eco.drop(columns=cols[0])

# train / test split
X_train_eco, X_test_eco, y_train_eco, y_test_eco = train_test_split(
   X_eco, y_eco, test_size=0.2, random_state=42
)

# ============================
# 3. Business: X, y 분리
# ============================
X_bus = df_business.drop(columns=['price', 'class'])
y_bus = df_business['price']

# 상수 컬럼 제거
nunique_bus = X_bus.nunique()
constant_cols_bus = nunique_bus[nunique_bus <= 1].index
X_bus = X_bus.drop(columns=constant_cols_bus)

# 더미변수 중복 제거
for prefix in prefixes:
    cols = [c for c in X_bus.columns if c.startswith(prefix)]
    if len(cols) > 0:
        X_bus = X_bus.drop(columns=cols[0])

# train / test split
X_train_bus, X_test_bus, y_train_bus, y_test_bus = train_test_split(
    X_bus, y_bus, test_size=0.2, random_state=42
 )

print("Economy train/test:", X_train_eco.shape, X_test_eco.shape)
print("Business train/test:", X_train_bus.shape, X_test_bus.shape)

Economy train/test: (165332, 45) (41334, 45)
Business train/test: (74789, 42) (18698, 42)


In [15]:
# ============================
# Linear Regression 
# ============================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

def run_linear(X_train, X_test, y_train, y_test, label):

    # 모델
    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Train 예측
    y_train_pred = model.predict(X_train)

    # Train / Test 점수
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    print("Train R2:", round(train_r2, 4))
    print("Test R2:", round(test_r2, 4))

    # 평가
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- [{label}] Linear Regression ---")
    print("R2:", round(r2, 4))
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))

    # 계수
    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': model.coef_
    }).sort_values(by='Coefficient', ascending=False)

    print(f"\n[{label} 중요 변수 Top 5]")
    print(coef_df.head(5))


# 실행
run_linear(X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy")
run_linear(X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business")

Train R2: 0.5579
Test R2: 0.5586

--- [Economy] Linear Regression ---
R2: 0.5586
MAE: 0.2728
RMSE: 0.349

[Economy 중요 변수 Top 5]
                  Feature  Coefficient
0                   stops     0.278156
6         airline_Vistara     0.226968
35    route_Kolkata_Delhi     0.176567
34  route_Kolkata_Chennai     0.157479
37   route_Kolkata_Mumbai     0.153962
Train R2: 0.6063
Test R2: 0.6086

--- [Business] Linear Regression ---
R2: 0.6086
MAE: 0.1339
RMSE: 0.1759

[Business 중요 변수 Top 5]
                   Feature  Coefficient
0                    stops     0.596724
3          airline_Vistara     0.149394
14  route_Bangalore_Mumbai     0.135523
32     route_Kolkata_Delhi     0.131998
35  route_Mumbai_Bangalore     0.128970


In [17]:
# ============================
# Lasso
# ============================

from sklearn.linear_model import Lasso

def run_lasso(X_train, X_test, y_train, y_test, label):

    model = Lasso(alpha=0.01)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Train 예측
    y_train_pred = model.predict(X_train)

    # Train / Test 점수
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    print("Train R2:", round(train_r2, 4))
    print("Test R2:", round(test_r2, 4))

    # 평가
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- [{label}] Lasso ---")
    print("R2:", round(r2, 4))
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))

    coef_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Coefficient': model.coef_
    })

    coef_df = coef_df[coef_df['Coefficient'] != 0]
    coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

    print(f"\n[{label} 선택된 변수 Top 5]")
    print(coef_df.head(5))


# 실행
run_lasso(X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy")
run_lasso(X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business")

Train R2: 0.489
Test R2: 0.4907

--- [Economy] Lasso ---
R2: 0.4907
MAE: 0.295
RMSE: 0.3749

[Economy 선택된 변수 Top 5]
                   Feature  Coefficient
0                    stops     0.235288
6          airline_Vistara     0.174367
9   departure_time_Morning     0.026877
11    arrival_time_Evening     0.017833
1                 duration     0.014970
Train R2: 0.5113
Test R2: 0.5118

--- [Business] Lasso ---
R2: 0.5118
MAE: 0.1508
RMSE: 0.1964

[Business 선택된 변수 Top 5]
           Feature  Coefficient
0            stops     0.480341
3  airline_Vistara     0.113417
1         duration     0.002856
2        days_left    -0.001472


In [39]:
# ============================
# Random Forest
# ============================

from sklearn.ensemble import RandomForestRegressor


def run_rf(data, label):

    X = data.drop(columns=['price', 'class'])
    y = data['price']

    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index
    X = X.drop(columns=constant_cols)

    prefixes = ['airline_', 'route_', 'departure_time_', 'arrival_time_']
    for prefix in prefixes:
        cols = [c for c in X.columns if c.startswith(prefix)]
        if len(cols) > 0:
            X = X.drop(columns=cols[0])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    train_score = model.score(X_train, y_train)

    y_pred = model.predict(X_test)

    # Train 예측
    y_train_pred = model.predict(X_train)

    # Train / Test 점수
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    print("Train R2:", round(train_r2, 4))
    print("Test R2:", round(test_r2, 4))

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- [{label}] Random Forest ---")
    print("R2:", round(r2, 4))
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))

    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    print(f"\n[{label} 중요 변수 Top 5]")
    print(importance_df.head(5))


# 실행
df_economy = df[df['class'] == 0]
df_business = df[df['class'] == 1]

run_rf(df_economy, "Economy")
run_rf(df_business, "Business")

Train R2: 0.9838
Test R2: 0.9002

--- [Economy] Random Forest ---
R2: 0.9002
MAE: 0.0881
RMSE: 0.1659

[Economy 중요 변수 Top 5]
            Feature  Importance
2         days_left    0.438686
1          duration    0.263874
6   airline_Vistara    0.049158
3  airline_GO_FIRST    0.017594
0             stops    0.016091
Train R2: 0.9858
Test R2: 0.9242

--- [Business] Random Forest ---
R2: 0.9242
MAE: 0.037
RMSE: 0.0774

[Business 중요 변수 Top 5]
                  Feature  Importance
1                duration    0.661925
3         airline_Vistara    0.066680
2               days_left    0.053935
8    arrival_time_Evening    0.013529
6  departure_time_Morning    0.009496


In [19]:
from sklearn.ensemble import RandomForestRegressor


def run_rf(X_train, X_test, y_train, y_test, label):

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Train 예측
    y_train_pred = model.predict(X_train)

    # Train / Test 점수
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_pred)

    print("Train R2:", round(train_r2, 4))
    print("Test R2:", round(test_r2, 4))

    # 평가
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- [{label}] Random Forest ---")
    print("R2:", round(r2, 4))
    print("MAE:", round(mae, 4))
    print("RMSE:", round(rmse, 4))

    importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    print(f"\n[{label} 중요 변수 Top 5]")
    print(importance_df.head(5))


# 실행
run_rf(X_train_eco, X_test_eco, y_train_eco, y_test_eco, "Economy")
run_rf(X_train_bus, X_test_bus, y_train_bus, y_test_bus, "Business")

Train R2: 0.9838
Test R2: 0.9002

--- [Economy] Random Forest ---
R2: 0.9002
MAE: 0.0881
RMSE: 0.1659

[Economy 중요 변수 Top 5]
            Feature  Importance
2         days_left    0.438686
1          duration    0.263874
6   airline_Vistara    0.049158
3  airline_GO_FIRST    0.017594
0             stops    0.016091
Train R2: 0.9858
Test R2: 0.9242

--- [Business] Random Forest ---
R2: 0.9242
MAE: 0.037
RMSE: 0.0774

[Business 중요 변수 Top 5]
                  Feature  Importance
1                duration    0.661925
3         airline_Vistara    0.066680
2               days_left    0.053935
8    arrival_time_Evening    0.013529
6  departure_time_Morning    0.009496
